# Isoform Enumeration Inspection

This notebook inspects the isoform enumeration pipeline (tautomers, protonation states, neutralization).
It runs enumeration on diverse test molecules, highlights known edge cases, and validates all output SMILES.

In [1]:
import pandas as pd
from rdkit import Chem, RDLogger

from golem.config import IsoformConfig
from golem.isoforms import (
    _enumerate_protonation,
    _enumerate_tautomers,
    _neutralize,
    enumerate_isoforms,
)

# Suppress RDKit warnings in notebook output
RDLogger.DisableLog('rdApp.*')

## 1. Test Molecules

A diverse set covering acids, bases, amides, heterocycles, and drug-like molecules.

In [2]:
test_molecules = {
    "Acetic acid": "CC(=O)O",
    "Ethylamine": "CCN",
    "Glycine": "NCC(=O)O",
    "N-acetylpiperidine (tertiary amide)": "CC(=O)N1CCCCC1",
    "Indole": "c1ccc2[nH]ccc2c1",
    "Pyrrole": "c1cc[nH]c1",
    "Guanine (tautomers)": "O=c1[nH]c2[nH]cnc2c(=O)[nH]1",
    "Phenol": "Oc1ccccc1",
    "Aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen": "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
}

print(f"Testing {len(test_molecules)} molecules")

Testing 10 molecules


## 2. Run Isoform Enumeration

In [3]:
config = IsoformConfig()
print(f"Config: max_tautomers={config.max_tautomers}, max_protomers={config.max_protomers}, ph_range={config.ph_range}")

results = []
for name, smi in test_molecules.items():
    mol = Chem.MolFromSmiles(smi)
    isoforms = enumerate_isoforms(smi, config)

    # Per-type breakdown
    n_taut, n_prot, n_neut = 0, 0, 0
    taut_capped, prot_capped = False, False

    if mol is not None:
        original_can = Chem.MolToSmiles(mol, canonical=True)

        # Tautomers
        taut_mols = _enumerate_tautomers(mol, max_tautomers=config.max_tautomers)
        taut_smiles = {Chem.MolToSmiles(t, canonical=True) for t in taut_mols}
        n_taut = len(taut_smiles)
        taut_capped = len(taut_mols) >= config.max_tautomers

        # Protonation states
        prot_mols = _enumerate_protonation(
            original_can, ph_range=config.ph_range, max_protomers=config.max_protomers,
        )
        prot_smiles = {Chem.MolToSmiles(p, canonical=True) for p in prot_mols}
        n_prot = len(prot_smiles)
        prot_capped = len(prot_mols) >= config.max_protomers

        # Neutralization
        neut_mols = _neutralize(mol)
        neut_smiles = {Chem.MolToSmiles(n, canonical=True) for n in neut_mols}
        n_neut = len(neut_smiles)

    results.append({
        "name": name,
        "parent_smiles": smi,
        "n_isoforms": len(isoforms),
        "isoforms": isoforms,
        "n_tautomers": n_taut,
        "taut_capped": taut_capped,
        "n_protomers": n_prot,
        "prot_capped": prot_capped,
        "n_neutralized": n_neut,
    })

df = pd.DataFrame(results)
df[["name", "parent_smiles", "n_isoforms"]]

Config: max_tautomers=10, max_protomers=10, ph_range=(6.4, 8.4)


,name,parent_smiles,n_isoforms
0,Acetic acid,CC(=O)O,3
1,Ethylamine,CCN,2
2,Glycine,NCC(=O)O,5
3,N-acetylpiperidine (tertiary amide),CC(=O)N1CCCCC1,2
4,Indole,c1ccc2[nH]ccc2c1,1
5,Pyrrole,c1cc[nH]c1,1
6,Guanine (tautomers),O=c1[nH]c2[nH]cnc2c(=O)[nH]1,18
7,Phenol,Oc1ccccc1,2
8,Aspirin,CC(=O)Oc1ccccc1C(=O)O,3
9,Ibuprofen,CC(C)Cc1ccc(C(C)C(=O)O)cc1,3


## 3. Per-Type Enumeration Breakdown

Counts of unique SMILES returned by each enumeration step.
An asterisk (**\***) indicates the raw enumeration hit the configured maximum, meaning additional forms may exist.

In [4]:
def _fmt(count, capped):
    """Format count with * suffix if capped at max."""
    return f"{count}*" if capped else str(count)

type_rows = []
for r in results:
    type_rows.append({
        "Molecule": r["name"],
        "Tautomers": _fmt(r["n_tautomers"], r["taut_capped"]),
        "Protomers": _fmt(r["n_protomers"], r["prot_capped"]),
        "Neutralized": str(r["n_neutralized"]),
        "Total (deduped)": str(r["n_isoforms"]),
    })

type_df = pd.DataFrame(type_rows)
print(type_df.to_string(index=False))
print(f"\n* = hit configured max (max_tautomers={config.max_tautomers}, max_protomers={config.max_protomers})")
print("  Counts are unique SMILES per step; Total is deduplicated across all steps.")

# Report capped molecules
capped = []
for r in results:
    if r["taut_capped"]:
        capped.append((r["name"], "tautomers", config.max_tautomers))
    if r["prot_capped"]:
        capped.append((r["name"], "protomers", config.max_protomers))

if capped:
    print(f"\nMolecules hitting enumeration limits:")
    for name, typ, limit in capped:
        print(f"  {name}: {typ} (max={limit})")
else:
    print("\nNo molecules hit enumeration limits.")

                           Molecule Tautomers Protomers Neutralized Total (deduped)
                        Acetic acid         2         1           1               3
                         Ethylamine         1         2           1               2
                            Glycine         3         2           1               5
N-acetylpiperidine (tertiary amide)         2         1           1               2
                             Indole         1         1           1               1
                            Pyrrole         1         1           1               1
                Guanine (tautomers)        9*       10*           1              18
                             Phenol         1         2           1               2
                            Aspirin         2         1           1               3
                          Ibuprofen         2         1           1               3

* = hit configured max (max_tautomers=10, max_protomers=10)
  Counts are un

## 4. Detailed Isoform Listing

In [5]:
for _, row in df.iterrows():
    print(f"\n{'='*60}")
    print(f"{row['name']}  ({row['parent_smiles']})")
    print(f"  {row['n_isoforms']} isoforms:")
    for i, iso in enumerate(row['isoforms']):
        tag = " (original)" if i == 0 else ""
        print(f"    [{i}] {iso}{tag}")


Acetic acid  (CC(=O)O)
  3 isoforms:
    [0] CC(=O)O (original)
    [1] C=C(O)O
    [2] CC(=O)[O-]

Ethylamine  (CCN)
  2 isoforms:
    [0] CCN (original)
    [1] CC[NH3+]

Glycine  (NCC(=O)O)
  5 isoforms:
    [0] NCC(=O)O (original)
    [1] NC=C(O)O
    [2] N=CC(O)O
    [3] [NH3+]CC(=O)[O-]
    [4] NCC(=O)[O-]

N-acetylpiperidine (tertiary amide)  (CC(=O)N1CCCCC1)
  2 isoforms:
    [0] CC(=O)N1CCCCC1 (original)
    [1] C=C(O)N1CCCCC1

Indole  (c1ccc2[nH]ccc2c1)
  1 isoforms:
    [0] c1ccc2[nH]ccc2c1 (original)

Pyrrole  (c1cc[nH]c1)
  1 isoforms:
    [0] c1cc[nH]c1 (original)

Guanine (tautomers)  (O=c1[nH]c2[nH]cnc2c(=O)[nH]1)
  18 isoforms:
    [0] O=c1[nH]c(=O)c2nc[nH]c2[nH]1 (original)
    [1] O=c1[nH]c(O)nc2[nH]cnc12
    [2] O=c1[nH]c(=O)c2[nH]cnc2[nH]1
    [3] O=c1nc(O)c2nc[nH]c2[nH]1
    [4] O=c1nc(O)[nH]c2[nH]cnc12
    [5] O=c1nc2[nH]cnc2c(O)[nH]1
    [6] O=c1[nH]c2ncnc-2c(O)[nH]1
    [7] O=c1[nH]c(O)nc2nc[nH]c12
    [8] O=c1nc(O)c2[nH]cnc2[nH]1
    [9] O=c1[n-]c(=O)c2nc[n-]

## 5. Edge Case Inspection

### 5a. Tertiary Amide — should NOT be protonated

In [6]:
amide_smi = "CC(=O)N1CCCCC1"
protomers = _enumerate_protonation(amide_smi, ph_range=(6.4, 8.4), max_protomers=10)
print(f"Tertiary amide protomers ({len(protomers)}):")
for pmol in protomers:
    psmi = Chem.MolToSmiles(pmol, canonical=True)
    has_charged_n = any(
        a.GetAtomicNum() == 7 and a.GetFormalCharge() > 0 for a in pmol.GetAtoms()
    )
    flag = " *** PROTONATED N (bad) ***" if has_charged_n else " (ok)"
    print(f"  {psmi}{flag}")

Tertiary amide protomers (1):
  CC(=O)N1CCCCC1 (ok)


### 5b. Nitrogen with 4+ Hydrogens — should be filtered

In [7]:
# Check that no protomer in our results has N with >= 4 H
n4h_count = 0
for _, row in df.iterrows():
    for iso in row['isoforms']:
        mol = Chem.MolFromSmiles(iso)
        if mol is None:
            continue
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() == 7 and atom.GetTotalNumHs() >= 4:
                print(f"  N(4+H) found in: {iso} (parent: {row['parent_smiles']})")
                n4h_count += 1

if n4h_count == 0:
    print("No N(4+H) found in any isoform output.")
else:
    print(f"\nWARNING: {n4h_count} N(4+H) instances found!")

No N(4+H) found in any isoform output.


## 6. Validate All Output SMILES

In [8]:
total = 0
valid = 0
invalid = []

for _, row in df.iterrows():
    for iso in row['isoforms']:
        total += 1
        mol = Chem.MolFromSmiles(iso)
        if mol is not None:
            try:
                Chem.SanitizeMol(mol)
                valid += 1
            except Exception:
                invalid.append((row['name'], iso))
        else:
            invalid.append((row['name'], iso))

print(f"Total isoforms: {total}")
print(f"Valid: {valid}")
print(f"Invalid: {len(invalid)}")
if invalid:
    print("\nInvalid SMILES:")
    for name, smi in invalid:
        print(f"  {name}: {smi}")

Total isoforms: 40
Valid: 40
Invalid: 0


## 7. Summary Statistics

In [9]:
total_parents = len(df)
total_isoforms = df['n_isoforms'].sum()
mean_expansion = df['n_isoforms'].mean()
max_expansion = df.loc[df['n_isoforms'].idxmax()]

print(f"Parents: {total_parents}")
print(f"Total isoforms: {total_isoforms}")
print(f"Mean expansion: {mean_expansion:.1f}x")
print(f"Max expansion: {max_expansion['n_isoforms']}x ({max_expansion['name']})")
print(f"Failures (invalid SMILES): {len(invalid)}")
print(f"\nExpansion per molecule:")
for _, row in df.iterrows():
    bar = '#' * row['n_isoforms']
    print(f"  {row['name']:40s} {row['n_isoforms']:3d}  {bar}")

Parents: 10
Total isoforms: 40
Mean expansion: 4.0x
Max expansion: 18x (Guanine (tautomers))
Failures (invalid SMILES): 0

Expansion per molecule:
  Acetic acid                                3  ###
  Ethylamine                                 2  ##
  Glycine                                    5  #####
  N-acetylpiperidine (tertiary amide)        2  ##
  Indole                                     1  #
  Pyrrole                                    1  #
  Guanine (tautomers)                       18  ##################
  Phenol                                     2  ##
  Aspirin                                    3  ###
  Ibuprofen                                  3  ###


## 8. Tautomer Comparison: MolVS (Gypsum-DL) vs RDKit Native

Compare tautomer counts and SMILES sets produced by MolVS `TautomerEnumerator` (used internally by Gypsum-DL) against the RDKit native `rdMolStandardize.TautomerEnumerator`.

In [10]:
from rdkit.Chem.MolStandardize import rdMolStandardize

MOLVS_AVAILABLE = False
try:
    from molvs import tautomer as molvs_tautomer
    MOLVS_AVAILABLE = True
    print("MolVS available ✓")
except ImportError:
    print("MolVS not installed — MolVS columns will be empty")


def _enumerate_tautomers_molvs(mol, max_tautomers=10):
    """Enumerate tautomers using MolVS TautomerEnumerator."""
    if not MOLVS_AVAILABLE:
        return []
    try:
        m = Chem.RemoveHs(mol)
        Chem.Kekulize(m)
        enum = molvs_tautomer.TautomerEnumerator(max_tautomers=max_tautomers)
        tauts = list(enum.enumerate(m))
        results = []
        for t in tauts:
            try:
                Chem.SanitizeMol(t)
                results.append(Chem.MolToSmiles(t, canonical=True))
            except Exception:
                continue
        return sorted(set(results))
    except Exception as e:
        print(f"  MolVS error: {e}")
        return []


def _enumerate_tautomers_rdkit(mol, max_tautomers=10):
    """Enumerate tautomers using RDKit native TautomerEnumerator."""
    try:
        enumerator = rdMolStandardize.TautomerEnumerator()
        enumerator.SetMaxTautomers(max_tautomers)
        tauts = list(enumerator.Enumerate(mol))
        return sorted({Chem.MolToSmiles(t, canonical=True) for t in tauts})
    except Exception as e:
        print(f"  RDKit error: {e}")
        return []


print(f"Defined helpers: _enumerate_tautomers_molvs, _enumerate_tautomers_rdkit")

MolVS available ✓
Defined helpers: _enumerate_tautomers_molvs, _enumerate_tautomers_rdkit


In [11]:
comparison_data = []

for name, smi in test_molecules.items():
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue

    rdkit_tauts = _enumerate_tautomers_rdkit(mol)
    molvs_tauts = _enumerate_tautomers_molvs(mol)

    rdkit_set = set(rdkit_tauts)
    molvs_set = set(molvs_tauts)
    common = rdkit_set & molvs_set
    rdkit_only = rdkit_set - molvs_set
    molvs_only = molvs_set - rdkit_set

    comparison_data.append({
        "Molecule": name,
        "RDKit count": len(rdkit_set),
        "MolVS count": len(molvs_set),
        "Common": len(common),
        "RDKit-only": len(rdkit_only),
        "MolVS-only": len(molvs_only),
        "_rdkit_set": rdkit_set,
        "_molvs_set": molvs_set,
    })

comp_df = pd.DataFrame(comparison_data)
print(comp_df[["Molecule", "RDKit count", "MolVS count", "Common", "RDKit-only", "MolVS-only"]].to_string(index=False))

                           Molecule  RDKit count  MolVS count  Common  RDKit-only  MolVS-only
                        Acetic acid            2            2       2           0           0
                         Ethylamine            1            1       1           0           0
                            Glycine            3            3       3           0           0
N-acetylpiperidine (tertiary amide)            2            2       2           0           0
                             Indole            1            1       1           0           0
                            Pyrrole            1            1       1           0           0
                Guanine (tautomers)           10           14      10           0           4
                             Phenol            2            1       1           1           0
                            Aspirin            2            2       2           0           0
                          Ibuprofen            2            

### Detailed Per-Molecule Tautomers

In [12]:
for row in comparison_data:
    name = row["Molecule"]
    rdkit_set = row["_rdkit_set"]
    molvs_set = row["_molvs_set"]
    common = rdkit_set & molvs_set
    rdkit_only = rdkit_set - molvs_set
    molvs_only = molvs_set - rdkit_set

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"  Common ({len(common)}):")
    for s in sorted(common):
        print(f"    {s}")
    if rdkit_only:
        print(f"  RDKit-only ({len(rdkit_only)}):")
        for s in sorted(rdkit_only):
            print(f"    {s}")
    if molvs_only:
        print(f"  MolVS-only ({len(molvs_only)}):")
        for s in sorted(molvs_only):
            print(f"    {s}")
    if not rdkit_only and not molvs_only:
        print(f"  (identical)")


Acetic acid
  Common (2):
    C=C(O)O
    CC(=O)O
  (identical)

Ethylamine
  Common (1):
    CCN
  (identical)

Glycine
  Common (3):
    N=CC(O)O
    NC=C(O)O
    NCC(=O)O
  (identical)

N-acetylpiperidine (tertiary amide)
  Common (2):
    C=C(O)N1CCCCC1
    CC(=O)N1CCCCC1
  (identical)

Indole
  Common (1):
    c1ccc2[nH]ccc2c1
  (identical)

Pyrrole
  Common (1):
    c1cc[nH]c1
  (identical)

Guanine (tautomers)
  Common (10):
    O=c1[nH]c(=O)c2[nH]cnc2[nH]1
    O=c1[nH]c(=O)c2nc[nH]c2[nH]1
    O=c1[nH]c(O)nc2[nH]cnc12
    O=c1[nH]c(O)nc2nc[nH]c12
    O=c1[nH]c2ncnc-2c(O)[nH]1
    O=c1nc(O)[nH]c2[nH]cnc12
    O=c1nc(O)c2nc[nH]c2[nH]1
    O=c1nc2[nH]cnc2c(O)[nH]1
    Oc1nc(O)c2nc[nH]c2n1
    Oc1nc2ncnc-2c(O)[nH]1
  MolVS-only (4):
    O=c1nc(O)[nH]c2nc[nH]c12
    O=c1nc(O)c2[nH]cnc2[nH]1
    O=c1nc2nc[nH]c2c(O)[nH]1
    Oc1nc(O)c2ncnc-2[nH]1

Phenol
  Common (1):
    Oc1ccccc1
  RDKit-only (1):
    O=C1C=CC=CC1

Aspirin
  Common (2):
    C=C(O)Oc1ccccc1C(=O)O
    CC(=O)Oc1ccccc1C

### Validation

Verify that all generated tautomers from both methods are valid SMILES.

In [13]:
total_checked = 0
invalid_tauts = []

for row in comparison_data:
    name = row["Molecule"]
    for source, smiles_set in [("RDKit", row["_rdkit_set"]), ("MolVS", row["_molvs_set"])]:
        for smi in smiles_set:
            total_checked += 1
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                invalid_tauts.append((name, source, smi, "parse failed"))
                continue
            try:
                Chem.SanitizeMol(mol)
            except Exception as e:
                invalid_tauts.append((name, source, smi, str(e)))

print(f"Checked {total_checked} tautomer SMILES across both methods")
if invalid_tauts:
    print(f"INVALID tautomers ({len(invalid_tauts)}):")
    for name, source, smi, reason in invalid_tauts:
        print(f"  [{source}] {name}: {smi} — {reason}")
else:
    print("All tautomers are valid SMILES.")

Checked 55 tautomer SMILES across both methods
All tautomers are valid SMILES.


### Summary

Overall comparison statistics for MolVS vs RDKit tautomer enumeration.

In [14]:
total_rdkit = sum(r["RDKit count"] for r in comparison_data)
total_molvs = sum(r["MolVS count"] for r in comparison_data)
total_common = sum(r["Common"] for r in comparison_data)
total_rdkit_only = sum(r["RDKit-only"] for r in comparison_data)
total_molvs_only = sum(r["MolVS-only"] for r in comparison_data)
n_identical = sum(1 for r in comparison_data if r["RDKit-only"] == 0 and r["MolVS-only"] == 0)

print(f"Molecules tested:           {len(comparison_data)}")
print(f"Identical results:          {n_identical}/{len(comparison_data)}")
print(f"Total RDKit tautomers:      {total_rdkit}")
print(f"Total MolVS tautomers:      {total_molvs}")
print(f"Common tautomers:           {total_common}")
print(f"RDKit-only tautomers:       {total_rdkit_only}")
print(f"MolVS-only tautomers:       {total_molvs_only}")
print()
if total_rdkit_only > 0:
    print("RDKit produces additional tautomers not found by MolVS.")
    print("MolVS is slightly more conservative (filters aromaticity-breaking forms).")
elif total_molvs_only > 0:
    print("MolVS produces additional tautomers not found by RDKit.")
else:
    print("Both methods produce identical tautomer sets for all test molecules.")

Molecules tested:           10
Identical results:          8/10
Total RDKit tautomers:      26
Total MolVS tautomers:      29
Common tautomers:           25
RDKit-only tautomers:       1
MolVS-only tautomers:       4

RDKit produces additional tautomers not found by MolVS.
MolVS is slightly more conservative (filters aromaticity-breaking forms).
